# Modelado — Laboratorio 3: Deep Learning ASL (versión PyTorch)

Este notebook continúa a partir de `preprocesamiento.ipynb`. Carga los tensores ya guardados
(`X_train.npy`, `y_train.npy`, etc.) y desarrolla los ejercicios 4 al 9 del laboratorio, usando
**PyTorch** en vez de TensorFlow (porque TensorFlow todavía no tiene soporte para Python 3.14):

1. Dos CNN (básica y profunda)
2. Red fully-connected simple
3. Otro algoritmo (Random Forest)
4. Comparación de todos los modelos
5. Data augmentation
6. Prueba con fotos propias
7. Reflexión de accesibilidad y sesgo

> **Instalación necesaria (una sola vez):** en tu entorno con Python 3.14 corre en terminal:
> `pip install torch torchvision`
> (esto sí tiene wheels para 3.14 a diferencia de tensorflow).

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

# Cargar los tensores generados en preprocesamiento.ipynb
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_val   = np.load("X_val.npy")
y_val   = np.load("y_val.npy")
X_test  = np.load("X_test.npy")
y_test  = np.load("y_test.npy")

with open("id2label.pkl", "rb") as f:
    id2label = pickle.load(f)
with open("label2id.pkl", "rb") as f:
    label2id = pickle.load(f)

NUM_CLASSES = len(np.unique(y_train))
IMG_SIZE = X_train.shape[1]

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Clases:", NUM_CLASSES)


Usando dispositivo: cpu
Train: (13195, 64, 64, 3) Val: (2827, 64, 64, 3) Test: (2828, 64, 64, 3)
Clases: 29


## Dataset y DataLoaders de PyTorch

PyTorch espera las imágenes en formato `(canales, alto, ancho)` en vez de `(alto, ancho, canales)`
como numpy/cv2, así que hacemos la transposición dentro del `Dataset`.

In [3]:
class ASLDataset(Dataset):
    def __init__(self, X, y, transform=None):
        # de (N, H, W, C) a (N, C, H, W), float32
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2)
        self.y = torch.tensor(y, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        img = self.X[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.y[idx]

BATCH_SIZE = 64

train_ds = ASLDataset(X_train, y_train)
val_ds   = ASLDataset(X_val, y_val)
test_ds  = ASLDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


## Función de entrenamiento y evaluación (reutilizable para todos los modelos)

In [4]:
def entrenar_modelo(model, train_loader, val_loader, epochs=15, lr=1e-3):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        # --- entrenamiento ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total += xb.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # --- validación ---
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                loss = criterion(out, yb)
                val_running_loss += loss.item() * xb.size(0)
                val_correct += (out.argmax(1) == yb).sum().item()
                val_total += xb.size(0)

        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} - loss: {train_loss:.4f} - acc: {train_acc:.4f} "
              f"- val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")

    return history


def predecir(model, loader):
    model.eval()
    preds, reales = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            out = model(xb)
            preds.extend(out.argmax(1).cpu().numpy())
            reales.extend(yb.numpy())
    return np.array(reales), np.array(preds)


def evaluar_modelo(nombre, y_true, y_pred, class_names=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"=== {nombre} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision (macro): {prec:.4f}")
    print(f"Recall (macro):    {rec:.4f}")
    print(f"F1-score (macro):  {f1:.4f}\n")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Matriz de confusión — {nombre}")
    plt.xlabel("Predicho")
    plt.ylabel("Real")
    plt.show()

    return {"modelo": nombre, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

resultados = []  # aquí acumulamos el resumen de cada modelo para la tabla comparativa final
class_names = [id2label[i] for i in range(NUM_CLASSES)]
